In [16]:
import pandas as pd

In [17]:
import os
import pickle

In [18]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error

In [19]:
import mlflow
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1774702997462, experiment_id='1', last_update_time=1774702997462, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, workspace='default'>

In [20]:
def read_dataframe(filename):
    if filename.endswith('.csv'):
        df = pd.read_csv(filename)

        
    elif filename.endswith('.parquet'):
        df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    


    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']

    return df

In [21]:
df_train = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet')
len(df_train), len(df_val)

(73908, 73908)

In [22]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [23]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

#Using Xgboost

In [24]:
import xgboost as xgb


In [25]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)


In [28]:

with mlflow.start_run():
    
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
        'learning_rate': 0.09585355369315604,
        'max_depth': 30,
        'min_child_weight': 1.060597050922164,
        'objective': 'reg:linear',
        'reg_alpha': 0.018060244040060163,
        'reg_lambda': 0.011658731377413597,
        'seed': 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

    

a:\AI\mlops\orchestration\orchestration-env\Lib\site-packages\xgboost\callback.py:385: UserWarning: [15:15:24] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\objective\regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()


[0]	validation-rmse:10.72329
[1]	validation-rmse:9.98335
[2]	validation-rmse:9.33340
[3]	validation-rmse:8.76577
[4]	validation-rmse:8.26821
[5]	validation-rmse:7.83661
[6]	validation-rmse:7.46124
[7]	validation-rmse:7.13859
[8]	validation-rmse:6.86118
[9]	validation-rmse:6.62295
[10]	validation-rmse:6.42046
[11]	validation-rmse:6.24749
[12]	validation-rmse:6.09840
[13]	validation-rmse:5.97248
[14]	validation-rmse:5.86594
[15]	validation-rmse:5.77398
[16]	validation-rmse:5.69477
[17]	validation-rmse:5.62847
[18]	validation-rmse:5.56995
[19]	validation-rmse:5.51934
[20]	validation-rmse:5.47588
[21]	validation-rmse:5.43829
[22]	validation-rmse:5.40642
[23]	validation-rmse:5.37794
[24]	validation-rmse:5.35367
[25]	validation-rmse:5.33047
[26]	validation-rmse:5.31008
[27]	validation-rmse:5.29199
[28]	validation-rmse:5.27658
[29]	validation-rmse:5.26235
[30]	validation-rmse:5.25017
[31]	validation-rmse:5.23849
[32]	validation-rmse:5.22905
[33]	validation-rmse:5.21875
[34]	validation-rmse:5.

2026/03/28 15:16:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run sincere-robin-783 at: http://localhost:5000/#/experiments/1/runs/ebda6866bb3847d6a9ca3465a09b4978
🧪 View experiment at: http://localhost:5000/#/experiments/1
